# Track B — Cyclone Track, Intensity & Per-Horizon Intensification Warnings
### v3: intensification warnings now computed separately at +6h, +12h, and +24h

Previously the intensification warning was always computed at +24h
internally. This version trains a dedicated intensification model at
EACH horizon, with a threshold-in-knots that scales to the time window
(a +5kt change in 6h is roughly as significant as +10kt in 24h), and its
own tuned decision threshold per horizon.

**No changes to the input CSV are needed** — everything here is built
from columns already in `ibtracs_NI_cleaned.csv`.

**Before running:** update `DATA_PATH` in Step 1.

## Step 1 — Install dependencies and load the cleaned dataset

In [1]:
# !pip install pandas scikit-learn joblib

import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, mean_absolute_error, classification_report,
    precision_score, recall_score, fbeta_score, confusion_matrix,
)

In [2]:
# --- EDIT THIS to point at your local cleaned CSV ---
DATA_PATH = "ibtracs_NI_cleaned.csv"

raw = pd.read_csv(DATA_PATH)
print(f"Rows: {len(raw)}, storms: {raw['SID'].nunique()}")
raw.head()

Rows: 11576, storms: 435


,SID,SEASON,NAME,ISO_TIME,NATURE,LAT,LON,WIND_KTS,PRES_MB,CATEGORY,DIST2LAND,LANDFALL,STORM_SPEED,STORM_DIR
0,1981283N07132,1981,FABIAN,1981-10-10 18:00:00,MX,9.1,129.9,NaN,1004.0,NaN,395,352,9,285
1,1981283N07132,1981,FABIAN,1981-10-11 00:00:00,MX,9.2,129.1,NaN,1004.0,NaN,311,261,9,285
2,1981283N07132,1981,FABIAN,1981-10-11 06:00:00,MX,9.7,128.0,NaN,1004.0,NaN,205,153,13,295
3,1981283N07132,1981,FABIAN,1981-10-11 12:00:00,MX,10.2,126.8,NaN,1004.0,NaN,117,80,13,290
4,1981283N07132,1981,FABIAN,1981-10-11 18:00:00,MX,10.6,125.4,NaN,1004.0,NaN,21,0,15,290


## Step 2 — Feature engineering (base + acceleration)

In [3]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(["SID", "ISO_TIME"]).copy()
    df["ISO_TIME"] = pd.to_datetime(df["ISO_TIME"])
    g = df.groupby("SID")

    df["WIND_DELTA_1"] = g["WIND_KTS"].diff()
    df["PRES_DELTA_1"] = g["PRES_MB"].diff()
    df["STORM_AGE_H"] = (df["ISO_TIME"] - g["ISO_TIME"].transform("min")).dt.total_seconds() / 3600
    df["WIND_TREND_3"] = g["WIND_KTS"].transform(lambda s: s.diff().rolling(3, min_periods=1).mean())
    df["PRES_TREND_3"] = g["PRES_MB"].transform(lambda s: s.diff().rolling(3, min_periods=1).mean())
    df["WIND_TREND_6"] = g["WIND_KTS"].transform(lambda s: s.diff().rolling(6, min_periods=2).mean())
    df["PRES_TREND_6"] = g["PRES_MB"].transform(lambda s: s.diff().rolling(6, min_periods=2).mean())

    df["STORM_SPEED"] = pd.to_numeric(df["STORM_SPEED"], errors="coerce")
    df["DIST2LAND"] = pd.to_numeric(df["DIST2LAND"], errors="coerce")
    return df

def add_acceleration(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["WIND_ACCEL"] = df["WIND_TREND_3"] - df["WIND_TREND_6"]
    df["PRES_ACCEL"] = df["PRES_TREND_3"] - df["PRES_TREND_6"]
    return df

BASE_FEATURES = [
    "LAT", "LON", "WIND_KTS", "PRES_MB",
    "WIND_DELTA_1", "PRES_DELTA_1",
    "WIND_TREND_3", "PRES_TREND_3",
    "WIND_TREND_6", "PRES_TREND_6",
    "STORM_AGE_H", "STORM_SPEED", "DIST2LAND",
]
ACCEL_FEATURES = BASE_FEATURES + ["WIND_ACCEL", "PRES_ACCEL"]

feat = add_acceleration(engineer_features(raw))
feat[ACCEL_FEATURES].describe()

,LAT,LON,WIND_KTS,PRES_MB,WIND_DELTA_1,PRES_DELTA_1,WIND_TREND_3,PRES_TREND_3,WIND_TREND_6,PRES_TREND_6,STORM_AGE_H,STORM_SPEED,DIST2LAND,WIND_ACCEL,PRES_ACCEL
count,11576.000000,11576.000000,10041.000000,9553.000000,9413.000000,9098.000000,9849.000000,9304.000000,9665.000000,9062.000000,11576.000000,11571.00000,11576.000000,9286.000000,8872.000000
mean,16.423488,82.424672,38.394881,992.919397,0.110273,-0.088591,0.218855,-0.130482,0.329959,-0.195279,59.497581,7.61343,258.724862,-0.176131,0.073540
std,5.203233,14.656913,20.359100,13.369969,3.726913,2.551882,3.141157,1.982979,2.399761,1.603586,55.262508,4.64981,267.667202,1.492314,1.088893
min,1.400000,42.800000,3.000000,890.000000,-49.000000,-25.000000,-34.666667,-18.333333,-20.000000,-15.833333,0.000000,0.00000,0.000000,-18.333333,-11.666667
25%,12.400000,72.575000,25.000000,990.000000,0.000000,0.000000,0.000000,-0.666667,0.000000,-0.666667,21.000000,5.00000,0.000000,-0.833333,-0.166667
50%,16.300000,84.000000,30.000000,996.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,45.000000,7.00000,184.000000,0.000000,0.000000
75%,20.800000,88.225000,45.000000,1001.000000,0.000000,0.000000,1.666667,0.333333,1.333333,0.166667,84.000000,10.00000,447.000000,0.000000,0.333333
max,31.000000,156.300000,140.000000,1012.000000,60.000000,55.000000,60.000000,25.000000,27.500000,13.333333,516.000000,54.00000,1560.000000,11.000000,13.333333


## Step 3 — Category + position targets at each horizon (unchanged)

In [4]:
CATEGORY_COLLAPSE = {
    "Depression": "Depression/DD",
    "Deep Depression": "Depression/DD",
    "Cyclonic Storm": "Cyclonic Storm",
    "Severe Cyclonic Storm": "Severe+",
    "Very Severe Cyclonic Storm": "Severe+",
    "Extremely Severe Cyclonic Storm": "Severe+",
    "Super Cyclonic Storm": "Severe+",
}
HORIZONS = {"+6h": 2, "+12h": 4, "+24h": 8}
REGRESSOR_HORIZONS = {"+12h", "+24h"}

def build_category_target(df: pd.DataFrame, horizon_steps: int) -> pd.DataFrame:
    df = df.copy()
    df["CATEGORY_GROUPED"] = df["CATEGORY"].map(CATEGORY_COLLAPSE)
    g = df.groupby("SID")
    df["TARGET_CATEGORY"] = g["CATEGORY_GROUPED"].shift(-horizon_steps)
    df["TARGET_LAT"] = g["LAT"].shift(-horizon_steps)
    df["TARGET_LON"] = g["LON"].shift(-horizon_steps)
    return df

def storm_level_split(X, y, groups, test_size=0.2, seed=42):
    splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    train_idx, test_idx = next(splitter.split(X, y, groups))
    return X.iloc[train_idx], X.iloc[test_idx], y.iloc[train_idx], y.iloc[test_idx]

def prepare_category(df: pd.DataFrame):
    cls_df = df.dropna(subset=ACCEL_FEATURES + ["TARGET_CATEGORY"])
    reg_df = df.dropna(subset=ACCEL_FEATURES + ["TARGET_LAT", "TARGET_LON"])
    return (
        (cls_df[ACCEL_FEATURES], cls_df["TARGET_CATEGORY"], cls_df["SID"]),
        (reg_df[ACCEL_FEATURES], reg_df[["TARGET_LAT", "TARGET_LON"]], reg_df["SID"]),
    )

## Step 4 — Train category classifiers + position regressors

In [5]:
category_results = {}

for label, steps in HORIZONS.items():
    targeted = build_category_target(feat, steps)
    (X_cls, y_cls, g_cls), (X_reg, y_reg, g_reg) = prepare_category(targeted)

    Xc_tr, Xc_te, yc_tr, yc_te = storm_level_split(X_cls, y_cls, g_cls)
    sc = StandardScaler().fit(Xc_tr)
    clf = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42)
    clf.fit(sc.transform(Xc_tr), yc_tr)
    preds_c = clf.predict(sc.transform(Xc_te))
    bal_acc = balanced_accuracy_score(yc_te, preds_c)
    print(f"{label} classification -> balanced accuracy: {bal_acc:.3f}")

    result = {"clf": clf, "clf_scaler": sc, "balanced_accuracy": bal_acc}

    if label in REGRESSOR_HORIZONS:
        Xr_tr, Xr_te, yr_tr, yr_te = storm_level_split(X_reg, y_reg, g_reg)
        sr = StandardScaler().fit(Xr_tr)
        reg = RandomForestRegressor(n_estimators=300, random_state=42)
        reg.fit(sr.transform(Xr_tr), yr_tr)
        result.update({"reg": reg, "reg_scaler": sr})

    category_results[label] = result

+6h classification -> balanced accuracy: 0.888
+12h classification -> balanced accuracy: 0.822
+24h classification -> balanced accuracy: 0.651


## Step 5 — Per-horizon intensification targets (new)

The "counts as meaningful intensification" bar scales with the horizon —
treating +10kt as equally significant whether it happens in 6h or 24h
would be an unfair comparison, since the same change is far more dramatic
over a shorter window.

| Horizon | Threshold |
|---|---|
| +6h  | +5kt  |
| +12h | +7kt  |
| +24h | +10kt |

In [6]:
INTENSIFY_THRESHOLD_KT = {"+6h": 5, "+12h": 7, "+24h": 10}

def build_intensify_target(df: pd.DataFrame, horizon_steps: int, kt_threshold: int) -> pd.DataFrame:
    df = df.copy()
    g = df.groupby("SID")
    df["FUTURE_WIND"] = g["WIND_KTS"].shift(-horizon_steps)
    df["WILL_INTENSIFY"] = (df["FUTURE_WIND"] >= df["WIND_KTS"] + kt_threshold).astype(int)
    return df

def prepare_intensify(df: pd.DataFrame):
    df = df.dropna(subset=ACCEL_FEATURES + ["WILL_INTENSIFY"])
    return df[ACCEL_FEATURES], df["WILL_INTENSIFY"], df["SID"]

## Step 6 — Train and threshold-tune an intensification model per horizon

For each horizon: train the binary classifier, sweep the probability
decision threshold, and pick a practical operating point. Note: at +6h
and +12h, the F2 score (which favors recall) kept improving all the way
to the lowest threshold tested — but that came with precision dropping
to ~30%, meaning roughly 2 false alarms per real hit. Rather than chase
that to the edge, we picked thresholds that keep precision above ~40% at
every horizon, since an alert system with too many false alarms stops
being trusted.

In [7]:
PROB_THRESHOLDS = [0.5, 0.4, 0.35, 0.3, 0.25, 0.2, 0.15, 0.1]

# Chosen operating points, based on the sweep (see notebook discussion above)
DEPLOYED_THRESHOLD = {"+6h": 0.25, "+12h": 0.20, "+24h": 0.15}

intensify_results = {}

for label, steps in HORIZONS.items():
    kt = INTENSIFY_THRESHOLD_KT[label]
    targeted = build_intensify_target(feat, steps, kt)
    X, y, groups = prepare_intensify(targeted)

    X_tr, X_te, y_tr, y_te = storm_level_split(X, y, groups)
    scaler = StandardScaler().fit(X_tr)
    clf = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42)
    clf.fit(scaler.transform(X_tr), y_tr)
    probs = clf.predict_proba(scaler.transform(X_te))[:, 1]

    t = DEPLOYED_THRESHOLD[label]
    preds = (probs >= t).astype(int)
    recall = recall_score(y_te, preds, zero_division=0)
    precision = precision_score(y_te, preds, zero_division=0)
    print(f"{label} (+{kt}kt, threshold={t}) -> recall={recall:.3f}, precision={precision:.3f}")

    intensify_results[label] = {"clf": clf, "scaler": scaler, "threshold": t}

+6h (+5kt, threshold=0.25) -> recall=0.853, precision=0.337
+12h (+7kt, threshold=0.2) -> recall=0.841, precision=0.348
+24h (+10kt, threshold=0.15) -> recall=0.928, precision=0.405


## Step 7 — Save all trained models

In [9]:
import os
os.makedirs("models", exist_ok=True)

for label, r in category_results.items():
    tag = label.replace("+", "")
    joblib.dump(r["clf"], f"models/track_b_classifier_{tag}.joblib")
    joblib.dump(r["clf_scaler"], f"models/track_b_scaler_{tag}.joblib")
    if "reg" in r:
        joblib.dump(r["reg"], f"models/track_b_regressor_{tag}.joblib")
        joblib.dump(r["reg_scaler"], f"models/track_b_reg_scaler_{tag}.joblib")

for label, r in intensify_results.items():
    tag = label.replace("+", "")
    joblib.dump(r["clf"], f"models/track_b_intensify_clf_{tag}.joblib")
    joblib.dump(r["scaler"], f"models/track_b_intensify_scaler_{tag}.joblib")

print("Saved all models to ./models/")

Saved all models to ./models/


## Step 8 — Updated inference function

`intensification_warning` is now computed using the model and threshold
that match the REQUESTED horizon, instead of always using +24h.

In [10]:
def predict_next(history_df: pd.DataFrame, horizon: str = "+12h") -> dict:
    """
    history_df: chronological recent timesteps for ONE storm (needs 6-7+ rows),
                with columns SID, ISO_TIME, LAT, LON, WIND_KTS, PRES_MB,
                STORM_SPEED, DIST2LAND.
    horizon: one of "+6h", "+12h", "+24h"
    """
    if horizon not in HORIZONS:
        raise ValueError(f"horizon must be one of {list(HORIZONS)}")

    tag = horizon.replace("+", "")
    clf = joblib.load(f"models/track_b_classifier_{tag}.joblib")
    scaler = joblib.load(f"models/track_b_scaler_{tag}.joblib")

    df = history_df.copy()
    df["SID"] = "LIVE"
    engineered = add_acceleration(engineer_features(df))
    features = engineered.iloc[[-1]][ACCEL_FEATURES]
    if features.isna().any(axis=None):
        raise ValueError("Not enough history to compute features (need 6-7+ prior timesteps).")

    features_s = scaler.transform(features)
    result = {"category": clf.predict(features_s)[0], "horizon": horizon}

    if horizon in REGRESSOR_HORIZONS:
        reg = joblib.load(f"models/track_b_regressor_{tag}.joblib")
        reg_scaler = joblib.load(f"models/track_b_reg_scaler_{tag}.joblib")
        lat, lon = reg.predict(reg_scaler.transform(features))[0]
        result["predicted_lat"] = round(float(lat), 2)
        result["predicted_lon"] = round(float(lon), 2)
    else:
        result["predicted_lat"] = None
        result["predicted_lon"] = None
        result["position_note"] = "Not offered at +6h (no better than naive persistence)"

    # Intensification warning now matches the REQUESTED horizon
    int_clf = joblib.load(f"models/track_b_intensify_clf_{tag}.joblib")
    int_scaler = joblib.load(f"models/track_b_intensify_scaler_{tag}.joblib")
    prob = int_clf.predict_proba(int_scaler.transform(features))[0, 1]
    result["intensification_warning"] = bool(prob >= DEPLOYED_THRESHOLD[horizon])
    result["intensification_probability"] = round(float(prob), 3)
    result["intensification_threshold_kt"] = INTENSIFY_THRESHOLD_KT[horizon]

    return result

# Smoke test on a real storm from the dataset
sample_sid = raw["SID"].iloc[-200]
sample_history = raw[raw["SID"] == sample_sid].sort_values("ISO_TIME").head(8)

for h in ["+6h", "+12h", "+24h"]:
    print(h, predict_next(sample_history, horizon=h))

+6h {'category': 'Depression/DD', 'horizon': '+6h', 'predicted_lat': None, 'predicted_lon': None, 'position_note': 'Not offered at +6h (no better than naive persistence)', 'intensification_warning': False, 'intensification_probability': 0.187, 'intensification_threshold_kt': 5}
+12h {'category': 'Depression/DD', 'horizon': '+12h', 'predicted_lat': 20.99, 'predicted_lon': 67.52, 'intensification_warning': False, 'intensification_probability': 0.007, 'intensification_threshold_kt': 7}
+24h {'category': 'Cyclonic Storm', 'horizon': '+24h', 'predicted_lat': 21.88, 'predicted_lon': 66.67, 'intensification_warning': True, 'intensification_probability': 1.0, 'intensification_threshold_kt': 10}
